In the previous notebook we saw that Sliding Window Memory bounds cost by discarding old messages entirely. That hard cutoff means the agent doesn't even know it forgot something.

Summary Memory takes a different approach. Instead of discarding history, it compresses it. Think of it like reading a long book and writing a one-page summary at the end of each chapter. You can't quote the book word-for-word anymore, but you still know what happened. A secondary LLM call periodically condenses older messages into a running textual summary. The agent loses exact wording but retains the gist (key facts, decisions, and context) across arbitrarily long conversations.

The catch: summaries are lossy (they can't perfectly reconstruct the original). Each compression cycle can lose details, shift emphasis, or subtly distort facts. Over many cycles this summary drift compounds. The agent's "memory" can diverge from what actually happened.

By the end of this notebook you'll understand:

How to build a rolling-summary memory system from scratch 

The summarizer loop: when to trigger it, what prompt to use, and how the summary evolves.

How summaries drift over long conversations, with a controlled experiment and visualizations.

Practical mitigations for drift and information loss.

> **Where we are in the progression:**
>
> | Technique | Mechanism | Problem it solves | Problem it introduces |
> |---|---|---|---|
> | 01 — Buffer | Keep all messages | Perfect recall | Unbounded cost |
> | 02 — Sliding Window | Keep last N turns | Capped cost | Forgets early context |
> | **03 — Summary Memory** | **Compress old turns into a summary** | **Keeps meaning, not just recency** | **Lossy compression** |
>
> Summary Memory answers the question Technique 02 left open: *can we preserve the meaning of early turns without paying to keep them verbatim?*

---

## What Is Summary Memory?

### The core idea in one sentence
When the conversation grows too long, use the LLM itself to compress the oldest turns into a concise summary — then replace those raw turns with the summary in the context window.

---

### The mental model — a rolling newspaper

Imagine a newspaper that rewrites yesterday's news into a single paragraph each morning, then uses that paragraph as the starting point for today's edition. The full detail of yesterday is gone, but the meaning survives. Today's full articles are kept verbatim. Yesterday becomes a paragraph. Last week becomes a sentence.

```
Context window at Turn 10 (window size = 3, summary active):

  [System Prompt]
  [SUMMARY: Chiru earns ₹1,20,000/month, expenses ₹60,000,
   has a maturing FD of ₹50,000, is risk-averse, asked about
   SIPs and was advised to consider debt funds.]
  [Turn 8 — verbatim]
  [Turn 9 — verbatim]
  [Turn 10 — verbatim]
```

The model sees the summary of everything that happened in Turns 1–7, plus the last 3 turns in full. Critical facts from Turn 1 survive — not as raw text, but distilled into the summary.

---

### Point 1 — The summary is generated by the LLM, not hand-coded

You give GPT-4o the old turns and a prompt like:

> *"Summarise this financial advisory conversation, preserving all key facts: salary, expenses, assets, risk profile, and any advice given."*

The model produces a compact paragraph. That paragraph replaces the raw turns. This is called **abstractive summarisation** — the model generates new text that captures the essence, rather than extracting exact sentences.

---

### Point 2 — Summarisation is triggered by a threshold, not on every turn

You do not summarise after every message — that would be expensive and slow. You define a trigger:
- **Turn-based:** summarise when the buffer exceeds N turns
- **Token-based:** summarise when total tokens exceed a budget threshold

When the trigger fires, the oldest K turns are compressed into a summary and replaced. The recent turns stay verbatim.

---

### Point 3 — The summary is injected as a special context block, not as a message

The summary is not a user message or an assistant message. It is injected as a system-level context block — typically appended to the system prompt or inserted as a special `system` message at the top of the conversation. This signals to the model that the summary is background knowledge, not part of the dialogue.

---

### Point 4 — Lossy compression is the fundamental risk

Every summary is a lossy operation. Information is lost. The model decides what matters — and it can get that wrong. A rare but critical instruction like "the user is allergic to equity investments" might survive the first summary but vanish by the third compression cycle.

Research from 2026 (arxiv 2603.07670) confirms: *"A rare but critical instruction from day one may survive the first compression but is exactly the kind of low-frequency, high-importance detail that tends to vanish by the third pass."*

This is the core production risk of summary memory. The mitigation is a well-engineered summarisation prompt that explicitly lists the categories of facts to preserve.

---

### Point 5 — Progressive summarisation: summaries of summaries

In long-running sessions, you can chain summaries. When the summary itself grows large, it gets re-summarised into an even more compact form. This is called **progressive summarisation** or **hierarchical summarisation**.

```
Level 0: Raw turns (full verbatim — most tokens)
Level 1: Rolling summary (one paragraph per N turns)
Level 2: Session summary (one paragraph per session)
Level 3: User profile (key facts only — fewest tokens)
```

Each level trades detail for compression. Technique 15 (Memory Compaction) in this course builds this hierarchy fully. Here we implement Level 0 → Level 1.

---

## FinCoach Example

**Without summary memory (Technique 02 — the problem):**
```
Turn 6 — User: "Based on my salary, am I saving enough?"
FinCoach: "Without knowing your exact salary, it's hard to say..."
← Salary from Turn 1 was evicted. Agent forgot.
```

**With summary memory (Technique 03 — the fix):**
```
Turn 6 — User: "Based on my salary, am I saving enough?"
Context contains:
  [SUMMARY: User earns ₹1,20,000/month, expenses ₹60,000, surplus
   ₹60,000. Has FD ₹50,000 maturing in 3 months. Risk-averse.]
  + [Turns 4, 5, 6 verbatim]
FinCoach: "Based on your salary of ₹1,20,000 and expenses of ₹60,000,
           your ₹60,000 monthly surplus represents a 50% savings rate —
           well above the recommended 20-30%."
← Salary survived compression. Agent answered correctly.
```

---

## Trade-offs

| | |
|---|---|
| ✅ Preserves key facts across long sessions | ❌ Summarisation is lossy — details can be lost |
| ✅ Token cost stays bounded | ❌ Each summarisation call costs tokens and adds latency |
| ✅ Better than window for domain-critical facts | ❌ Summary quality depends on prompt engineering |
| ✅ Natural fit for multi-session continuity | ❌ Hard to audit — what exactly got summarised? |

## Production Verdict

> **Strong production pattern — widely used, but must be engineered carefully.**
>
> Summary memory is used in production by Claude Code, Kiro CLI, and most long-running agent frameworks. The risk is information loss on the third or fourth compression cycle. The mitigation is a domain-specific summarisation prompt that explicitly names the categories of facts that must survive — salary, risk profile, goals, decisions made. Never use a generic summary prompt in a domain with critical facts.

---

### Key Concepts

Rolling summary: A single text block that gets incrementally updated as the conversation grows. Each update folds new messages into the existing summary.

Summarizer prompt: The instruction given to the LLM to produce the summary. Its wording controls what's preserved (facts, decisions, tone) and what's discarded.

Refresh trigger: The rule that decides when to re-summarize. Common options: after every n messages, when the buffer exceeds a token threshold, or on every turn.

Summary drift: The gradual distortion of facts across repeated summarization cycles. Details get softened, merged, or lost entirely over time.

Compression ratio: How much shorter the summary is compared to the raw messages it replaces. Higher compression means more information loss.

Buffer zone: Recent messages kept word-for-word alongside the summary. This gives the LLM exact context for the latest exchanges while older context lives in the compressed summary.

Mermaid source sequenceDiagram

    participant U as User
    participant B as Message Buffer
    participant S as Summary Store
    participant LLM as Claude (Chat)
    participant SLLM as Claude (Summarizer)

    Note over S: Summary: "" (empty)
    Note over B: Buffer: []

    U->>B: msg 1
    B->>LLM: [summary="", msg1]
    LLM-->>B: msg 2
    Note over B: Buffer: [msg1, msg2]

    U->>B: msg 3
    B->>LLM: [summary="", msg1..msg3]
    LLM-->>B: msg 4
    Note over B: Buffer: [msg1..msg4] - trigger!

    rect rgb(255, 245, 230)
        Note over B,SLLM: 🔄 Summarization Cycle
        B->>SLLM: "Summarize: {old_summary} + {msg1..msg4}"
        SLLM-->>S: Updated summary
        B->>B: Clear buffer
    end

    U->>B: msg 5
    B->>LLM: [summary="...", msg5]
    LLM-->>B: msg 6
    Note over B: Buffer: [msg5, msg6]
    Note over S: Summary carries forward
compressed history

In [12]:
# --- Standard library ---
import json                              # Serialising session state to JSON.
import os                                # Reading environment variables.
import time                              # Rate-limit delays between API calls.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ compatible.
from typing import List, Dict, Optional  # Type hints.
from dataclasses import dataclass, field, asdict  # Clean data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.

In [13]:
# --- API Key Setup ---
# Option A: Colab Secrets (recommended — key never appears in notebook output)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("Key loaded from Colab Secrets.")
except Exception:
    # Option B: Environment variable (for local Jupyter / VS Code)
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets."

# Create the OpenAI client — reused for all API calls including summarisation.
client = openai.OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4o"
# gpt-4o: used for BOTH the FinCoach conversation AND the summarisation calls.
# In production: you may use a cheaper, faster model (gpt-4o-mini) for
# summarisation and reserve gpt-4o for the main conversation.
# This saves ~80% on summarisation cost with acceptable quality.

SUMMARY_MODEL = "gpt-4o-mini"
# Separate model for summarisation — cheaper, faster, sufficient quality.
# gpt-4o-mini: $0.15/M input tokens vs $2.50/M for gpt-4o.
# For a compression task (not a reasoning task), this is the right choice.

TOKENISER = tiktoken.get_encoding("o200k_base")
# o200k_base: the exact tokeniser for gpt-4o and gpt-4o-mini.

print(f"Main model    : {MODEL}")
print(f"Summary model : {SUMMARY_MODEL}")

Key loaded from environment variable.
Main model    : gpt-4o
Summary model : gpt-4o-mini


---
## Part 1 — The Message Data Model

Same as previous techniques. Consistent throughout the course.

In [14]:
@dataclass
class Message:
    """
    A single conversation message.
    Identical to Techniques 01 and 02 — the data model never changes.
    """
    role: str
    # 'user' or 'assistant' — who sent this message.

    content: str
    # The text of the message.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set when the Message is created.

    def to_api_format(self) -> Dict[str, str]:
        """Return only role and content — what the OpenAI API accepts."""
        return {"role": self.role, "content": self.content}

print("Message dataclass defined.")

Message dataclass defined.


---
## Part 2 — The Summarisation Engine

Before building the full memory class, we build the summarisation engine separately. This is good production design — the summariser is a distinct component, independently testable, swappable with a different model or strategy without touching the memory class.

In [15]:
# ─────────────────────────────────────────────────────────────────────────────
# THE SUMMARISATION PROMPT — the most important engineering decision in this
# entire technique. A generic prompt loses domain-critical facts.
# A domain-specific prompt explicitly lists what must survive compression.
# ─────────────────────────────────────────────────────────────────────────────

FINCOACH_SUMMARY_PROMPT = """You are summarising a financial advisory conversation for FinCoach.
Your summary will be used as the memory context for the NEXT API call.
The agent will use this summary — not the original turns — to continue advising the user.

CRITICAL: The following categories of facts MUST be preserved if mentioned.
Missing any of these is a failure:

1. FINANCIAL FACTS: salary, take-home income, expenses, savings rate, assets, liabilities, debts
2. INVESTMENT CONTEXT: existing investments (FDs, SIPs, stocks, property), their values and maturity dates
3. RISK PROFILE: stated risk tolerance (conservative, moderate, aggressive)
4. GOALS: short-term and long-term financial goals mentioned by the user
5. DECISIONS MADE: any investment or financial decisions the user confirmed
6. ADVICE GIVEN: key recommendations FinCoach made that the user acknowledged
7. PERSONAL CONTEXT: age, family situation, employer, any life events shared

Format:
- Write in third person ("The user...", "FinCoach advised...")
- Be concise but complete — every fact in the list above must appear if mentioned
- Do NOT add facts not present in the conversation
- Do NOT use bullet points — write as a single coherent paragraph
- Maximum 150 words
"""
# This prompt is deliberately long and explicit.
# Every category listed is a type of fact that would otherwise be
# silently dropped by a generic summariser.
# In production: treat this prompt as a versioned asset — changes to
# it affect summary quality for ALL users. Test changes carefully.


def summarise_turns(
    turns_to_summarise: List[Message],
    existing_summary: Optional[str] = None
) -> str:
    """
    Compress a list of conversation turns into a concise summary string.

    If there is already a summary from a previous compression cycle,
    it is passed in as context so the new summary can incorporate it.
    This is the mechanism for progressive summarisation:
    each new summary builds on the previous one.

    Args:
        turns_to_summarise: List of Message objects to compress.
        existing_summary:   Previous summary string, if any.
                            None on the first compression cycle.

    Returns:
        A summary string — a compressed representation of the input turns.
    """

    # Build the text block of turns to be summarised.
    turns_text = "\n".join(
        f"{msg.role.upper()}: {msg.content}"
        for msg in turns_to_summarise
    )
    # Format each message as "ROLE: content" on its own line.
    # Example: "USER: My salary is ₹1,20,000"
    #          "ASSISTANT: Great. How can I help?"

    # Build the user message for the summarisation call.
    if existing_summary:
        # Progressive summarisation: incorporate the existing summary
        # so we do not lose facts from previous compression cycles.
        user_content = (
            f"EXISTING SUMMARY (from previous turns):\n{existing_summary}\n\n"
            f"NEW TURNS TO ADD TO THE SUMMARY:\n{turns_text}\n\n"
            f"Produce an updated summary that incorporates BOTH the existing "
            f"summary AND the new turns. Do not lose any facts from either."
        )
    else:
        # First compression cycle — no existing summary to carry forward.
        user_content = (
            f"CONVERSATION TURNS TO SUMMARISE:\n{turns_text}\n\n"
            f"Produce a summary of these turns."
        )

    # Call the summary model — cheaper and faster than the main model.
    summary_response = client.chat.completions.create(
        model=SUMMARY_MODEL,
        # Using gpt-4o-mini for summarisation — 16x cheaper than gpt-4o.
        # Summarisation is a compression task, not a reasoning task.
        # Cheaper models perform well for this.

        max_tokens=300,
        # Cap the summary at 300 tokens.
        # Our prompt says max 150 words — 300 tokens gives comfortable headroom.
        # In production: enforce the word count in the prompt AND cap tokens.

        temperature=0.0,
        # Temperature 0 for summarisation — we want deterministic, factual output.
        # Creativity is undesirable here; accuracy is everything.

        messages=[
            {"role": "system", "content": FINCOACH_SUMMARY_PROMPT},
            # The domain-specific summarisation instructions.
            {"role": "user",   "content": user_content},
            # The turns to summarise, formatted as plain text.
        ]
    )

    summary_text = summary_response.choices[0].message.content.strip()
    # Extract the summary text and strip any leading/trailing whitespace.

    summary_tokens = len(TOKENISER.encode(summary_text))
    original_tokens = sum(len(TOKENISER.encode(m.content)) for m in turns_to_summarise)
    # Calculate compression ratio for logging.

    print(f"  [SUMMARISE] {len(turns_to_summarise)} messages ({original_tokens} tokens) "
          f"→ summary ({summary_tokens} tokens) | "
          f"compression: {original_tokens/max(summary_tokens,1):.1f}x")

    return summary_text
    # The caller stores this string and injects it into future API calls
    # in place of the raw turns that were just compressed.


print("Summarisation engine defined.")
print(f"Summary model : {SUMMARY_MODEL} (cheaper than {MODEL} for compression tasks)")

Summarisation engine defined.
Summary model : gpt-4o-mini (cheaper than gpt-4o for compression tasks)


---
## Part 3 — The SummaryMemory Class

The memory class manages three distinct zones:
1. **Summary zone** — compressed representation of old turns (injected into context)
2. **Recent turns zone** — last N turns kept verbatim (also in context)
3. **Archive zone** — complete session history (not in context, available for retrieval)

In [16]:
class SummaryMemory:
    """
    Manages conversation memory using a rolling summary.

    When the conversation exceeds `max_turns_before_summary`, the oldest
    `turns_to_summarise` turns are compressed into a summary string.
    The summary is injected into every subsequent API call as a context block.
    Recent turns are kept verbatim.

    Three zones:
    ┌─────────────────────────────────────────────────────────┐
    │  [Summary of Turns 1-7]  ← compressed, always in context│
    │  [Turn 8 verbatim]        ← recent, always in context   │
    │  [Turn 9 verbatim]        ← recent, always in context   │
    │  [Turn 10 verbatim]       ← recent, always in context   │
    └─────────────────────────────────────────────────────────┘
    Archive: [All 10 turns] ← not in context, available for search
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        max_turns_before_summary: int = 6,
        # Trigger summarisation when the recent buffer exceeds this many turns.
        # Default 6: after 6 turns accumulate, compress the oldest 4 into a summary.
        # In production for FinCoach: 8-10 is a good starting point.
        turns_to_keep_verbatim: int = 3,
        # After summarisation, keep this many recent turns verbatim.
        # The rest are compressed. Default 3 = keep the last 3 turns in full.
        # These are the turns the model needs for conversational coherence.
    ):
        self.session_id = session_id
        self.user_id = user_id
        self.system_prompt = system_prompt

        self.max_turns_before_summary = max_turns_before_summary
        # When recent_turns exceeds this, summarisation is triggered.

        self.turns_to_keep_verbatim = turns_to_keep_verbatim
        # How many of the most recent turns to keep in full after compression.
        # These recent turns provide conversational continuity.

        self.current_summary: Optional[str] = None
        # The current rolling summary string.
        # None until the first summarisation trigger fires.
        # After the first trigger: a string of max ~150 words.
        # After progressive summarisation: updated to include newer turns.

        self.recent_turns: List[Message] = []
        # The recent conversation turns kept verbatim.
        # Grows with each new turn. When it hits max_turns_before_summary,
        # the oldest turns_to_summarise turns get compressed into summary.

        self.archive: List[Message] = []
        # Complete session history — every message ever, in order.
        # Not sent to the API. Available for later retrieval (Technique 06).
        # In production: persist this to a database.

        self._total_turns = 0
        # Total completed turns ever — for analytics and audit.

        self._summary_count = 0
        # How many times summarisation has been triggered this session.
        # Useful for monitoring compression cycles and quality degradation risk.

        self.created_at = datetime.now(timezone.utc).isoformat()

        print(f"[SummaryMemory] Initialised — session: {self.session_id}, "
              f"user: {self.user_id}")
        print(f"  Trigger   : summarise after {self.max_turns_before_summary} turns")
        print(f"  Keep      : {self.turns_to_keep_verbatim} turns verbatim after compression")
        print(f"  Compress  : {self.max_turns_before_summary - self.turns_to_keep_verbatim} turns per cycle")

    # ------------------------------------------------------------------
    # CORE OPERATIONS
    # ------------------------------------------------------------------

    def add_message(self, role: str, content: str) -> None:
        """
        Add a new message to the conversation.
        After adding an assistant message (completing a turn),
        check whether the summarisation trigger should fire.
        """

        if role not in ("user", "assistant"):
            raise ValueError(f"Invalid role '{role}'. Use 'user' or 'assistant'.")

        new_message = Message(role=role, content=content)
        # Create the Message object with an auto-set UTC timestamp.

        self.recent_turns.append(new_message)
        # Add to the recent turns list — these are the verbatim turns.

        self.archive.append(new_message)
        # ALWAYS add to the archive — the complete session record.

        if role == "assistant":
            # An assistant message means a complete turn (user + assistant) is done.
            self._total_turns += 1

            current_turns = len(self.recent_turns) // 2
            # Convert message count to turn count for the threshold check.

            if current_turns >= self.max_turns_before_summary:
                # The recent buffer has hit the threshold — trigger summarisation.
                self._trigger_summarisation()
                # This compresses old turns into the rolling summary
                # and shrinks recent_turns back to turns_to_keep_verbatim.

    def _trigger_summarisation(self) -> None:
        """
        Internal method — called automatically when the turn threshold is hit.
        Compresses the oldest turns into the rolling summary.
        Keeps the most recent turns verbatim.
        """

        # Calculate how many messages to keep verbatim.
        # turns_to_keep_verbatim turns × 2 messages per turn.
        keep_messages = self.turns_to_keep_verbatim * 2

        # Split the recent_turns list into two parts.
        turns_to_compress = self.recent_turns[:-keep_messages]
        # Everything BEFORE the last keep_messages — these get compressed.
        # Example: if recent_turns has 12 messages and keep_messages=6,
        # turns_to_compress = messages [0:6] (oldest 3 turns).

        turns_to_keep = self.recent_turns[-keep_messages:]
        # Everything FROM the last keep_messages onward — these stay verbatim.
        # Example: messages [6:12] (most recent 3 turns).

        if not turns_to_compress:
            # Nothing to compress — this should not happen with correct settings,
            # but guard against it to prevent an empty summarisation call.
            return

        self._summary_count += 1
        # Increment the summary cycle counter before the API call.

        print(f"\n[SummaryMemory] Summarisation cycle #{self._summary_count} triggered")
        print(f"  Compressing {len(turns_to_compress)} messages "
              f"({len(turns_to_compress)//2} turns) into summary")
        print(f"  Keeping     {len(turns_to_keep)} messages "
              f"({len(turns_to_keep)//2} turns) verbatim")

        # Call the summarisation engine to compress the old turns.
        # Pass the existing summary so progressive summarisation works.
        self.current_summary = summarise_turns(
            turns_to_summarise=turns_to_compress,
            existing_summary=self.current_summary,
            # On cycle 1: existing_summary is None → fresh summary.
            # On cycle 2+: existing_summary is the previous summary →
            #   new summary incorporates both old summary and new turns.
        )

        # Replace recent_turns with only the verbatim portion.
        self.recent_turns = turns_to_keep
        # The compressed turns are now ONLY in:
        # 1. self.current_summary (as compressed text)
        # 2. self.archive (as full verbatim messages)
        # They are NO LONGER in self.recent_turns.

        print(f"  Summary preview: \"{self.current_summary[:100]}...\"")

    def get_messages_for_api(self) -> List[Dict[str, str]]:
        """
        Build the complete message list for the OpenAI API.

        Structure:
        1. System message (FinCoach's persona)
        2. Summary context block (if a summary exists) — injected as a system message
        3. Recent turns verbatim (last N turns)

        The summary is injected as a SYSTEM message (not user/assistant)
        to signal that it is background context, not dialogue.
        """

        messages = []
        # Start with an empty list and build it up in order.

        messages.append({"role": "system", "content": self.system_prompt})
        # 1. FinCoach's persona and instructions — always first.

        if self.current_summary:
            # 2. Inject the rolling summary as a second system message.
            # This is the key mechanism — the summary appears BEFORE
            # the recent turns so the model has it as background context.
            summary_block = (
                f"CONVERSATION SUMMARY (earlier turns):\n"
                f"{self.current_summary}\n"
                f"The conversation continues below."
            )
            messages.append({"role": "system", "content": summary_block})
            # Two system messages is valid in the OpenAI API.
            # The model treats them both as background instructions.

        for msg in self.recent_turns:
            messages.append(msg.to_api_format())
            # 3. Add the recent turns verbatim — these are the
            # last turns_to_keep_verbatim full turns.

        return messages
        # Final structure sent to the API:
        # [system: persona] + [system: summary?] + [recent turns]

    # ------------------------------------------------------------------
    # TOKEN COUNTING
    # ------------------------------------------------------------------

    def get_context_tokens(self) -> Dict[str, int]:
        """
        Break down the token count of the next API call by component.
        Returns a dict with tokens for each zone.
        Useful for monitoring and tuning the summarisation thresholds.
        """
        system_tokens  = len(TOKENISER.encode(self.system_prompt))
        # Constant overhead — same on every call.

        summary_tokens = len(TOKENISER.encode(self.current_summary)) \
                         if self.current_summary else 0
        # Summary tokens — grows slowly across compression cycles.
        # Ideally stays under 200 tokens with the 150-word cap.

        recent_tokens  = sum(
            len(TOKENISER.encode(msg.content)) for msg in self.recent_turns
        )
        # Recent turns — bounded by turns_to_keep_verbatim.
        # These reset after each summarisation cycle.

        return {
            "system": system_tokens,
            "summary": summary_tokens,
            "recent": recent_tokens,
            "total": system_tokens + summary_tokens + recent_tokens,
        }

    # ------------------------------------------------------------------
    # PERSISTENCE
    # ------------------------------------------------------------------

    def persist(self, filepath: str) -> None:
        """
        Save the full session state to a JSON file.
        Saves the summary, recent turns, archive, and all metadata.
        """
        tokens = self.get_context_tokens()

        record = {
            "schema_version": "1.0",
            "technique": "summary_memory",
            "session_id": self.session_id,
            "user_id": self.user_id,
            "model": MODEL,
            "summary_model": SUMMARY_MODEL,
            "created_at": self.created_at,
            "saved_at": datetime.now(timezone.utc).isoformat(),
            "config": {
                "max_turns_before_summary": self.max_turns_before_summary,
                "turns_to_keep_verbatim": self.turns_to_keep_verbatim,
            },
            "stats": {
                "total_turns": self._total_turns,
                "summary_cycles": self._summary_count,
                # How many times summarisation was triggered — quality risk indicator.
                # > 3 cycles: start auditing summary quality.
                "recent_turns_count": len(self.recent_turns) // 2,
                "archive_messages": len(self.archive),
                "context_tokens": tokens,
            },
            "current_summary": self.current_summary,
            # The rolling summary string — the core state to restore.
            "recent_turns": [asdict(m) for m in self.recent_turns],
            # Recent turns to restore in the verbatim buffer.
            "archive": [asdict(m) for m in self.archive],
            # Full session history — for compliance and long-term retrieval.
        }

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2, ensure_ascii=False)

        print(f"[SummaryMemory] Persisted — {self._total_turns} turns, "
              f"{self._summary_count} summary cycles, "
              f"{tokens['total']} context tokens")

    @classmethod
    def load(cls, filepath: str) -> "SummaryMemory":
        """Restore SummaryMemory from a saved JSON file."""

        with open(filepath, "r", encoding="utf-8") as f:
            record = json.load(f)

        if record.get("schema_version") != "1.0":
            raise ValueError(f"Unknown schema version: {record.get('schema_version')}")

        mem = cls(
            session_id=record["session_id"],
            user_id=record["user_id"],
            system_prompt="[LOADED — set system_prompt after loading]",
            max_turns_before_summary=record["config"]["max_turns_before_summary"],
            turns_to_keep_verbatim=record["config"]["turns_to_keep_verbatim"],
        )

        mem.current_summary = record["current_summary"]
        # Restore the summary — the most important state to recover.

        mem.recent_turns = [
            Message(role=m["role"], content=m["content"], timestamp=m["timestamp"])
            for m in record["recent_turns"]
        ]
        # Restore the verbatim recent turns.

        mem.archive = [
            Message(role=m["role"], content=m["content"], timestamp=m["timestamp"])
            for m in record["archive"]
        ]
        # Restore the full session history.

        mem._total_turns  = record["stats"]["total_turns"]
        mem._summary_count = record["stats"]["summary_cycles"]
        mem.created_at    = record["created_at"]

        print(f"[SummaryMemory] Loaded — {mem._total_turns} turns, "
              f"{mem._summary_count} summary cycles")
        if mem.current_summary:
            print(f"  Summary preview: \"{mem.current_summary[:80]}...\"")

        return mem

    # ------------------------------------------------------------------
    # DIAGNOSTICS
    # ------------------------------------------------------------------

    def print_stats(self) -> None:
        """Print a full breakdown of memory state and token usage."""
        tokens = self.get_context_tokens()

        print(f"\n{'='*62}")
        print(f"  Summary Memory Stats — Session: {self.session_id}")
        print(f"{'='*62}")
        print(f"  Total turns (all)      : {self._total_turns}")
        print(f"  Summary cycles fired   : {self._summary_count}")
        if self._summary_count > 2:
            print(f"  ⚠ WARNING: {self._summary_count} compression cycles may cause info loss")
            # After 3 cycles, start auditing summary quality manually.
        print(f"  Recent turns (verbatim): {len(self.recent_turns)//2}")
        print(f"  Archive (total msgs)   : {len(self.archive)}")
        print(f"")
        print(f"  Token breakdown for next API call:")
        print(f"    System prompt  : {tokens['system']:>6,} tokens")
        print(f"    Summary block  : {tokens['summary']:>6,} tokens")
        print(f"    Recent turns   : {tokens['recent']:>6,} tokens")
        print(f"    ─────────────────────────────")
        print(f"    TOTAL input    : {tokens['total']:>6,} tokens")
        print(f"")
        if self.current_summary:
            print(f"  Current summary:")
            print(f"  {self.current_summary}")
        else:
            print(f"  No summary yet — below the compression threshold.")
        print(f"{'='*62}\n")


print("SummaryMemory class defined.")

SummaryMemory class defined.


---
## Part 4 — The FinCoach Agent with Summary Memory

In [17]:
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use all context — including the conversation summary if provided — to give consistent, personalised advice."""
# Note the last line: explicitly instructs the model to use the summary context.
# Without this, the model may ignore the summary block in favour of recent turns.


def chat(
    memory: SummaryMemory,
    user_message: str,
    verbose: bool = True
) -> str:
    """
    Send one user message to FinCoach. Return the assistant's reply.
    Identical structure to Techniques 01 and 02 — only the memory object changes.
    """

    # STEP 1 — Add the user message.
    memory.add_message(role="user", content=user_message)
    # Adds to recent_turns and archive.
    # Does NOT trigger summarisation yet (that fires after the assistant reply).

    # STEP 2 — Call GPT-4o with the summary-augmented context.
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1024,
        temperature=0.7,
        messages=memory.get_messages_for_api(),
        # get_messages_for_api() returns:
        #   [system: persona]
        #   [system: summary block]  ← only if summary exists
        #   [recent turns verbatim]
        # The model sees the summary of old turns AND the full recent turns.
        # This is what allows it to correctly reference Turn 1 facts in Turn 10.
    )

    # STEP 3 — Extract the reply.
    assistant_reply = response.choices[0].message.content

    # STEP 4 — Add the reply and potentially trigger summarisation.
    memory.add_message(role="assistant", content=assistant_reply)
    # This adds to recent_turns and archive.
    # It also checks the turn threshold — if exceeded, _trigger_summarisation() fires.
    # So summarisation happens AFTER the response, not before — the user never waits.

    if verbose:
        tokens = memory.get_context_tokens()
        print(f"[Turn {memory._total_turns}] "
              f"API — prompt: {response.usage.prompt_tokens} tokens | "
              f"Context: sys={tokens['system']}, "
              f"summary={tokens['summary']}, "
              f"recent={tokens['recent']} | "
              f"Cycles: {memory._summary_count}")
        # Breaking down the token usage by zone shows exactly how the
        # summary is growing and the recent turns are cycling.

    return assistant_reply


print("FinCoach chat() function defined.")

FinCoach chat() function defined.


---
## Part 5 — Demo: The Forgetting Problem 


**Settings:** `max_turns_before_summary=4`, `turns_to_keep_verbatim=2`  
This means after 4 turns, the oldest 2 turns get compressed — summarisation fires during the demo.

In [18]:
summary_memory = SummaryMemory(
    session_id="session_sm_demo_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_turns_before_summary=4,
    # Summarise after 4 turns — fires during the 6-turn demo.
    turns_to_keep_verbatim=2,
    # Keep the most recent 2 turns verbatim after compression.
)

# THE SAME TURNS THAT BROKE TECHNIQUE 02
# Generic questions in turns 4-5 prevent context echo.
# In Technique 02 these caused Turn 6 to fail.
# With Summary Memory, the salary should survive in the summary.
demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    # Turn 1: salary — will be compressed after Turn 4.
    "My monthly expenses are about ₹60,000 for rent, food, and transport.",
    # Turn 2: expenses — will be compressed after Turn 4.
    "I have an FD of ₹50,000 that matures in 3 months. I am risk-averse.",
    # Turn 3: FD and risk profile.
    "What is a Systematic Investment Plan and how does it work?",
    # Turn 4: GENERIC QUESTION — no echo of salary.
    # SUMMARISATION FIRES AFTER THIS TURN.
    # Turns 1-2 get compressed into the summary.
    "What are the different types of mutual funds available in India?",
    # Turn 5: ANOTHER GENERIC QUESTION — no echo.
    "What is my exact monthly take-home salary that I told you at the start?",
    # Turn 6: DIRECT RECALL — this failed in Technique 02.
    # This time, the salary is in the summary. Watch the answer.
]

print("SUMMARY MEMORY DEMO — The Technique 02 Forgetting Problem, Fixed")
print("=" * 65)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=summary_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
print("Demo complete.")
summary_memory.print_stats()

[SummaryMemory] Initialised — session: session_sm_demo_001, user: chiru_001
  Trigger   : summarise after 4 turns
  Keep      : 2 turns verbatim after compression
  Compress  : 2 turns per cycle
SUMMARY MEMORY DEMO — The Technique 02 Forgetting Problem, Fixed

--- Turn 1 ---
User: Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.
[Turn 1] API — prompt: 166 tokens | Context: sys=136, summary=0, recent=158 | Cycles: 0
FinCoach: Hi Chiru! With a monthly take-home salary of ₹1,20,000, you're in a good position to start building a solid financial plan. A common budgeting rule is the 50/30/20 rule: allocate 50% of your income for essentials like rent and groceries, 30% for discretionary spending, and 20% for savings and investments. This means you could aim to save or invest ₹24,000 each month. Consider setting up an emergency fund covering 3-6 months of expenses and explore options like PPF, mutual funds, or SIPs for long-term growth. For personalized investment strategies, consultin

---
## Part 6 — Demonstrating Progressive Summarisation

Run a longer session — 10 turns — to trigger the summarisation cycle more than once and show progressive summarisation in action.

In [19]:
# Fresh memory for the progressive demo.
progressive_memory = SummaryMemory(
    session_id="session_sm_progressive_001",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    max_turns_before_summary=4,
    # Fire every 4 turns — will trigger twice in a 10-turn session.
    turns_to_keep_verbatim=2,
)

long_session_turns = [
    "Hi! I'm Chiru. My salary is ₹1,20,000 per month and expenses are ₹60,000.",
    "I'm 32 years old and risk-averse. My goal is to retire by 55.",
    "I have no investments right now except an FD of ₹50,000.",
    "What is the difference between PPF and NPS for retirement saving?",
    # ← FIRST SUMMARISATION TRIGGERS HERE (after Turn 4)
    "Which one would you recommend for someone who wants to retire by 55?",
    "What if I invest ₹10,000 per month in NPS — how much will I have at 55?",
    "That's good. Should I also keep some money in equity mutual funds?",
    "OK so split between NPS and equity SIP. How should I divide the ₹10,000?",
    # ← SECOND SUMMARISATION TRIGGERS HERE (after Turn 8)
    "One more thing — what about tax benefits on NPS and ELSS?",
    "OK can you summarise everything we decided today as a clear action plan?",
    # Turn 10: asks for a summary — model should use both summary cycles
    # AND the recent turns to produce a complete action plan.
]

print("PROGRESSIVE SUMMARISATION DEMO — 10 turns, 2 compression cycles expected")
print("=" * 65)

for i, user_message in enumerate(long_session_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = chat(memory=progressive_memory, user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 65)
progressive_memory.print_stats()
# Watch the summary_cycles counter reach 2.
# The second summary incorporates the first — progressive summarisation.

[SummaryMemory] Initialised — session: session_sm_progressive_001, user: chiru_001
  Trigger   : summarise after 4 turns
  Keep      : 2 turns verbatim after compression
  Compress  : 2 turns per cycle
PROGRESSIVE SUMMARISATION DEMO — 10 turns, 2 compression cycles expected

--- Turn 1 ---
User: Hi! I'm Chiru. My salary is ₹1,20,000 per month and expenses are ₹60,000.
[Turn 1] API — prompt: 172 tokens | Context: sys=136, summary=0, recent=239 | Cycles: 0
FinCoach: Hi Chiru! Based on your monthly salary of ₹1,20,000 and expenses of ₹60,000, you're left with a savings potential of ₹60,000 each month. Here are a few suggestions for managing this surplus:

1. **Emergency Fund**: Aim to build an emergency fund that covers 6 to 12 months of your expenses, which would be between ₹3,60,000 and ₹7,20,000. You could allocate a portion of your savings each month to achieve this.

2. **Investments**: Consider investing in a diversified portfolio, such as mutual funds or PPF, based on your risk tol

---
## The Summary Prompt Quality Test

This shows directly why the summarisation prompt matters. Same turns — generic prompt vs domain-specific prompt.

In [20]:
# Test the summarisation engine directly — no full conversation needed.
# This is how you validate summary quality in CI/CD.

test_turns = [
    Message(role="user",      content="My monthly take-home salary is ₹1,20,000."),
    Message(role="assistant", content="Got it. How can I help you with your finances?"),
    Message(role="user",      content="My monthly expenses are ₹60,000. I'm risk-averse and never invest in equity."),
    Message(role="assistant", content="Understood. With ₹60,000 surplus, low-risk options like FDs and debt funds suit your profile."),
    Message(role="user",      content="I have an FD of ₹50,000 maturing in 3 months and want to retire by 55."),
    Message(role="assistant", content="With your FD maturing soon and a retirement target of 55, PPF and NPS are worth considering."),
]

print("=" * 65)
print("SUMMARY QUALITY COMPARISON")
print("Same 6 messages, two different summarisation prompts")
print("=" * 65)

# --- Generic prompt (what most tutorials use) ---
GENERIC_SUMMARY_PROMPT = "Summarise this conversation briefly."

print("\n1. GENERIC PROMPT: 'Summarise this conversation briefly.'")
turns_text = "\n".join(f"{m.role.upper()}: {m.content}" for m in test_turns)

generic_response = client.chat.completions.create(
    model=SUMMARY_MODEL,
    max_tokens=200,
    temperature=0.0,
    messages=[
        {"role": "system", "content": GENERIC_SUMMARY_PROMPT},
        {"role": "user",   "content": turns_text}
    ]
)
generic_summary = generic_response.choices[0].message.content
print(f"Result: {generic_summary}")
print(f"Tokens: {len(TOKENISER.encode(generic_summary))}")

print()

# --- Domain-specific prompt (what we built) ---
print("2. DOMAIN-SPECIFIC PROMPT: explicitly lists salary, risk profile, constraints, goals")
domain_summary = summarise_turns(turns_to_summarise=test_turns, existing_summary=None)
print(f"Result: {domain_summary}")
print(f"Tokens: {len(TOKENISER.encode(domain_summary))}")

print("\n" + "=" * 65)
print("Key question: which summary preserves the equity constraint?")
print("'never invest in equity' — is it in the generic summary? Is it in the domain summary?")
print("That constraint could cause serious financial harm if lost.")

SUMMARY QUALITY COMPARISON
Same 6 messages, two different summarisation prompts

1. GENERIC PROMPT: 'Summarise this conversation briefly.'
Result: The user has a monthly take-home salary of ₹1,20,000 and expenses of ₹60,000, leaving a surplus of ₹60,000. They are risk-averse and do not invest in equity. The assistant suggests low-risk investment options like fixed deposits (FDs), debt funds, Public Provident Fund (PPF), and National Pension System (NPS) to help the user prepare for retirement by age 55.
Tokens: 89

2. DOMAIN-SPECIFIC PROMPT: explicitly lists salary, risk profile, constraints, goals
  [SUMMARISE] 6 messages (113 tokens) → summary (125 tokens) | compression: 0.9x
Result: The user has a monthly take-home salary of ₹1,20,000 and monthly expenses of ₹60,000, resulting in a surplus of ₹60,000. The user is risk-averse and does not invest in equity. They currently have a fixed deposit (FD) of ₹50,000 that will mature in 3 months and have a goal to retire by the age of 55. FinC

---
## Part 9 — Three-Way Comparison: Buffer vs Window vs Summary

In [21]:
# Illustrative — no API calls. Shows the structural comparison.

print("THREE-WAY COMPARISON — What the API receives at Turn 10")
print("=" * 65)

print("""
TECHNIQUE 01 — Conversation Buffer
────────────────────────────────────
API receives:
  [system: persona]
  [T1-user] [T1-ai] [T2-user] [T2-ai] ... [T10-user]
  ↑ ALL 10 turns verbatim

  Salary in context? ✅ YES — verbatim in T1-user
  Token cost?        ❌ GROWS every turn — T10 costs 10x T1
  Context overflow?  ❌ RISK at long sessions
""")

print("""
TECHNIQUE 02 — Sliding Window (size=5)
────────────────────────────────────────
API receives:
  [system: persona]
  [T6-user] [T6-ai] [T7-user] [T7-ai] ... [T10-user]
  ↑ Only last 5 turns — T1-T5 EVICTED

  Salary in context? ❌ NO — T1 was evicted at T6
  Token cost?        ✅ FIXED — same at T100 as at T5
  Context overflow?  ✅ NONE — size is capped
""")

print("""
TECHNIQUE 03 — Summary Memory (keep 3 verbatim)
─────────────────────────────────────────────────
API receives:
  [system: persona]
  [system: SUMMARY of T1-T7: Chiru earns ₹1,20,000, expenses ₹60,000,
   has FD ₹50,000 maturing Q3, risk-averse, never invests in equity,
   retirement goal age 55. Advised PPF and NPS.]
  [T8-user] [T8-ai] [T9-user] [T9-ai] [T10-user]
  ↑ Summary of old turns + last 3 turns verbatim

  Salary in context? ✅ YES — in the summary
  Token cost?        ✅ BOUNDED — summary + fixed recent window
  Context overflow?  ✅ NONE
  Constraint safe?   ✅ IF the summary prompt captures it
  Risk?              ⚠  Summary is lossy — quality depends on the prompt
""")

print("="*65)
print("Summary Memory is the best of buffer and window — but not free.")
print("The cost is engineering effort on the summarisation prompt.")
print("That prompt is mission-critical in regulated domains like finance.")

THREE-WAY COMPARISON — What the API receives at Turn 10

TECHNIQUE 01 — Conversation Buffer
────────────────────────────────────
API receives:
  [system: persona]
  [T1-user] [T1-ai] [T2-user] [T2-ai] ... [T10-user]
  ↑ ALL 10 turns verbatim

  Salary in context? ✅ YES — verbatim in T1-user
  Token cost?        ❌ GROWS every turn — T10 costs 10x T1
  Context overflow?  ❌ RISK at long sessions


TECHNIQUE 02 — Sliding Window (size=5)
────────────────────────────────────────
API receives:
  [system: persona]
  [T6-user] [T6-ai] [T7-user] [T7-ai] ... [T10-user]
  ↑ Only last 5 turns — T1-T5 EVICTED

  Salary in context? ❌ NO — T1 was evicted at T6
  Token cost?        ✅ FIXED — same at T100 as at T5
  Context overflow?  ✅ NONE — size is capped


TECHNIQUE 03 — Summary Memory (keep 3 verbatim)
─────────────────────────────────────────────────
API receives:
  [system: persona]
  [system: SUMMARY of T1-T7: Chiru earns ₹1,20,000, expenses ₹60,000,
   has FD ₹50,000 maturing Q3, risk-averse, n

---
## Part 10 — Session Persistence

In [22]:
SESSION_FILE = "/tmp/fincoach_sm_session_demo_001.json"
# Save the session from the demo in Part 5.

summary_memory.persist(SESSION_FILE)
# Saves current_summary, recent_turns, archive, and all stats.

print("\n--- Simulating service restart ---\n")

restored = SummaryMemory.load(SESSION_FILE)
# Restores the summary, recent turns, and archive from the file.

restored.system_prompt = FINCOACH_SYSTEM_PROMPT
# Set the system prompt after loading — it is not stored in the file.

print("\nContinuing after restore:")
follow_up = "Based on everything you know about my finances, what is my single biggest financial priority?"
print(f"User: {follow_up}")

response = chat(memory=restored, user_message=follow_up, verbose=True)
# The restored memory has the summary of Turns 1-4 and verbatim recent turns.
# FinCoach should answer using the salary, expenses, and FD from the summary.

print(f"FinCoach: {response}")

[SummaryMemory] Persisted — 6 turns, 2 summary cycles, 609 context tokens

--- Simulating service restart ---

[SummaryMemory] Initialised — session: session_sm_demo_001, user: chiru_001
  Trigger   : summarise after 4 turns
  Keep      : 2 turns verbatim after compression
  Compress  : 2 turns per cycle
[SummaryMemory] Loaded — 6 turns, 2 summary cycles
  Summary preview: "The user, Chiru, has a monthly take-home salary of ₹1,20,000 and monthly expense..."

Continuing after restore:
User: Based on everything you know about my finances, what is my single biggest financial priority?
[Turn 7] API — prompt: 671 tokens | Context: sys=136, summary=196, recent=407 | Cycles: 2
FinCoach: Based on your current financial situation, your single biggest financial priority should be establishing a robust emergency fund. Given your monthly expenses of ₹60,000, aim to save enough to cover at least 3 to 6 months' worth of expenses, which amounts to ₹1,80,000 to ₹3,60,000. This will provide you with a 

---
## Key Takeaways

**What you built:** A `SummaryMemory` class with a domain-specific summarisation engine, progressive summarisation across multiple cycles, session persistence, a fact-survival test, and a summary quality comparison.

---

### The complete progression

| | Buffer (01) | Window (02) | Summary (03) |
|---|---|---|---|
| Early context retained | ✅ All | ❌ Lost | ✅ Compressed |
| Token cost bounded | ❌ No | ✅ Yes | ✅ Yes |
| Context overflow risk | ❌ Yes | ✅ No | ✅ No |
| Information loss risk | ✅ None | ❌ Hard eviction | ⚠ Lossy compression |
| Implementation complexity | Low | Low | Medium |

---

### The three things to remember

1. **The summarisation prompt is the most important line of code in this entire technique.** A generic prompt loses domain-critical facts. A domain-specific prompt that explicitly lists what must survive is the difference between a working system and a liability — especially in regulated domains like financial services.

2. **Summarise AFTER the response, not before.** The user should never wait for a summarisation call to complete before getting their answer. The summarisation runs after `add_message(role='assistant')` — asynchronously in production.

3. **Monitor summary cycles in production.** After 3+ compression cycles, information loss compounds. Low-frequency but high-importance facts (constraints, hard rules, specific amounts) are at risk. A fact that survived cycle 1 may not survive cycle 3. The mitigation is the domain-specific prompt — and periodic human audits of summary quality.

---

### What Technique 04 fixes

Summary Memory compresses old turns but keeps recent turns verbatim. This means the token cost is: `summary tokens + recent turns tokens`. If recent turns are long (detailed advice, calculations), the recent zone can still be expensive. **Technique 04 — Summary Buffer Memory** combines a sliding window for the recent zone with a rolling summary for older turns — the hybrid that most production systems actually use.

---


### FAQ

Q: What is Summary Memory in agent memory?

A: Summary Memory replaces the raw conversation history with a running LLM-generated summary. After each turn, the model compresses all prior context into a short paragraph. This keeps token usage roughly constant (typically 200-500 tokens for the summary) regardless of conversation length. LangChain implements this as ConversationSummaryMemory. The cost is an extra LLM call per turn to update the summary, plus loss of verbatim detail from earlier messages.

Q: When should I use Summary Memory instead of Summary Buffer Memory?

A: Use pure Summary Memory when token budget is extremely tight and you do not need exact quotes from recent messages. Summary Buffer Memory (technique 04) keeps the last K messages verbatim alongside the summary, which doubles the memory overhead. If your model has a small context window (4k-8k tokens) and conversations run long (50+ turns), pure summarization gives you the most compression. Choose the buffer variant when recent-turn fidelity matters.

Q: What are the limits or failure modes of Summary Memory?

A: Summaries lose detail with every compression step. Specific numbers, names, and exact phrasing often get dropped or distorted. The quality depends on the summarization model: a weak model produces lossy or hallucinated summaries. Each turn requires an extra LLM call, adding latency (200-500ms) and cost. Over very long conversations, the summary can drift, emphasizing recent topics and forgetting earlier ones. There is no way to recover lost details once compressed.

Q: Can I combine Summary Memory with another memory technique?

A: Yes. Combine it with technique 06 (Vector Store Memory) for the best of both worlds. Write each message to a vector store before summarizing, so the summary handles general context and the vector store provides exact retrieval when needed. You can also layer it with technique 07 (Entity Memory): the summary tracks conversation flow while entity memory preserves structured facts about people, places, and things mentioned.

Q: What library or framework can I use to skip the implementation work?

A: LangChain provides ConversationSummaryMemory with configurable LLM backends and prompt templates. LlamaIndex supports summary-based chat memory through its ChatSummaryMemoryBuffer. Mem0 (technique 25) handles summarization automatically in its managed pipeline. Letta/MemGPT (technique 26) uses a similar approach in its core memory tier, compressing older context into archival storage. For a fully managed option, Zep (technique 27) runs asynchronous summarization server-side.

